# EvidGeo — full evaluation on all Drive data

Runs the corrected pipeline (`evidgeo/`, `scripts/run_eval.py`, `scripts/analyze.py`) on **every usable corpus in Drive**:

| Corpus | Drive folder | Images | Notes |
|---|---|---|---|
| v2 guidance | `multiguide_coco_v2/images/{sd14,sd15,biggan}/{text,i2i,weak}` | ~27.4k | SD: 512px, 50 steps, CFG 7.5 (text, i2i), **CFG 4.0 for weak** (prompt "a photo"), i2i strength 0.75. BigGAN-deep-256: truncation 0.4 ("text") vs 1.0 ("weak"). |
| v3 guidance | `multiguide_coco_v3/images/{sd/sd14,sd/sd15,biggan}/…` | ~10k | 1,249 new captions; local ids mapped to real COCO rows automatically. SD params not recorded in meta. |
| consistency | `multiguide_consistency/images/{text,i2i,weak}/<pid>_seed<k>.png` | 1,350 | 150 prompts × 3 seeds × 3 modes. Generator not recorded. |
| real COCO | Hugging Face `Erland/coco_captions_small` (train, shuffle 42) | one per prompt | Downloaded once and cached; paired to every generated image by row. |

Excluded on purpose: `multiguide_coco_v1` and `multiguide_coco_10`, whose text prompts were truncated to one character by a generator bug.

Every chunk is written to Drive as it finishes, so after a disconnect just re-run the same cell and it resumes. **Runtime: GPU (A100 if available).**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
REPO = '/content/evidgeocode'
if not os.path.exists(REPO):
    !git clone https://github.com/dpaul0501/evidgeocode.git {REPO}
else:
    !git -C {REPO} pull --ff-only
!pip -q install -r {REPO}/requirements.txt pyarrow statsmodels
%cd {REPO}
!python -m pytest -q tests

In [ ]:
D = '/content/drive/MyDrive'
V2, V3, CONS = f'{D}/multiguide_coco_v2', f'{D}/multiguide_coco_v3', f'{D}/multiguide_consistency'
MASTER = f'{V2}/cache/manifest_v3_N5000_seed42.jsonl'   # captions in HF shuffle order, rows 0..4999
OUT = f'{D}/evidgeo_eval/main'                            # results live on Drive
COMMON = f'--guidance-roots {V2} {V3} --consistency-root {CONS} --master-manifest {MASTER} --real'
for p in (V2, V3, CONS, MASTER):
    print(('ok  ' if os.path.exists(p) else 'MISSING ') + p)

## 1. Smoke test (8 images per bucket, ~2 min)
Check the printed counts: about 20 buckets and no warnings about missing captions.

In [ ]:
!python scripts/run_eval.py {COMMON} --out /content/smoke --limit-per-bucket 8 --faith-per-bucket 2 --grids 7
!python scripts/analyze.py --run-dir /content/smoke
import pandas as pd
pd.read_csv('/content/smoke/results/corpus_summary.csv')

## 2. Main run: every image, G = 7
LOO (49 masked passes per image), Grad-PEC, the prompt-swap control, the prompt-free saliency ablation, attention rollout, SpatialShap (exact PPR), Gaussian baselines, the α sweep, and deletion/insertion faithfulness on the first 300 prompts of each bucket.
Expect roughly 1–2 h on an A100. If Drive reads are the bottleneck (low img/s), run the staging cell below first.

In [ ]:
# Optional: copy images to local disk if img/s is low (Drive FUSE is slow for many small files).
# Outputs still go to Drive; only the inputs are read locally.
STAGE = False
if STAGE:
    roots = []
    for src in (V2, V3, CONS):
        dst = '/content/data/' + os.path.basename(src)
        !rsync -a --exclude 'paper_*' --exclude '*_outputs' --exclude 'empirical_v2_test' {src}/ {dst}/
        roots.append(dst)
    LV2, LV3, LCONS = roots
    COMMON = f'--guidance-roots {LV2} {LV3} --consistency-root {LCONS} --master-manifest {MASTER} --real'
print(COMMON)

In [ ]:
!python scripts/run_eval.py {COMMON} --out {OUT} --grids 7 --faith-per-bucket 300 --batch 32

## 3. Ablations (reviewer requests)
* **Patch size** G ∈ {5, 9, 14} on v2 (first 500 prompts per bucket), for R3.
* **Masking** zero vs blur instead of mean-colour fill (first 300 prompts per bucket): does masked-CLIP utility depend on the fill? (R3)

In [ ]:
!python scripts/run_eval.py {COMMON} --only multiguide_coco_v2 --out {D}/evidgeo_eval/grid --grids 5 9 14 --limit-per-bucket 500 --alpha-sweep
for mm in ('zero', 'blur'):
    !python scripts/run_eval.py {COMMON} --only multiguide_coco_v2 --out {D}/evidgeo_eval/mask_{mm} --mask-mode {mm} --limit-per-bucket 300 --alpha-sweep

## 4. Analysis → tables + `numbers.tex`
Every number in the paper should come from `results/numbers.tex` via `\\EV{key}`. Nothing is typed by hand.

In [ ]:
for run in ('main', 'grid', 'mask_zero', 'mask_blur'):
    !python scripts/analyze.py --run-dir {D}/evidgeo_eval/{run}
R = f'{OUT}/results'
display(pd.read_csv(f'{R}/corpus_summary.csv'))
display(pd.read_csv(f'{R}/guidance_tests.csv').query("signal in ['pec','rollout','ss']").round(4))
display(pd.read_csv(f'{R}/smoothing.csv').round(4))
display(pd.read_csv(f'{R}/faithfulness.csv').round(4))
display(pd.read_csv(f'{R}/stability_tests.csv').round(4))
display(pd.read_csv(f'{R}/duality.csv').round(4))
display(pd.read_csv(f'{R}/real_vs_generated.csv').query("signal=='pec'").round(4))
display(pd.read_csv(f'{R}/detection.csv').round(3))
display(pd.read_csv(f'{R}/prompt_dependence.csv').round(3))